# 07 - Swappable LLM Providers: OpenAI and Anthropic

A production RAG system shouldn't be locked into a single LLM provider. Reasons to support multiple providers:

- **Cost optimization** — different models have different price points
- **Capability matching** — Claude excels at long context, GPT-4 at function calling
- **Resilience** — if one API goes down, switch to another
- **Vendor lock-in avoidance** — good engineering practice

This notebook demonstrates our **LLM provider abstraction** — a factory pattern that lets you swap providers with a single parameter change.

In [ ]:
import os
import sys
import tempfile
import time

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

## The LLMProvider Factory

Our `LLMProvider` class wraps LangChain's `ChatOpenAI` and `ChatAnthropic` behind a unified interface:

In [ ]:
from rag_engine.llm.provider import LLMProvider

# Create OpenAI and Anthropic LLMs
openai_llm = LLMProvider.get_llm(provider="openai", model="gpt-4o-mini")
anthropic_llm = LLMProvider.get_llm(provider="anthropic", model="claude-sonnet-4-20250514")

print(f"OpenAI model:    {openai_llm.__class__.__name__} ({openai_llm.model_name})")
print(f"Anthropic model: {anthropic_llm.__class__.__name__} ({anthropic_llm.model})")

# Both implement the same interface (BaseChatModel)
from langchain_core.language_models import BaseChatModel

print(f"\nBoth are BaseChatModel? OpenAI={isinstance(openai_llm, BaseChatModel)}, Anthropic={isinstance(anthropic_llm, BaseChatModel)}")

## Build the Same RAG Pipeline with Both Providers

The key insight: everything else stays the same — same retriever, same prompt, same data. Only the LLM changes.

In [ ]:
from rag_engine.chains.rag_chain import build_rag_chain
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.loaders import load_documents
from rag_engine.retrieval.retriever import RetrieverFactory
from rag_engine.vectorstore.chroma_store import add_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

# Build knowledge base (same for both)
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
chunks = chunk_documents(md_docs + csv_docs, chunk_size=512, chunk_overlap=50)

temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb07", persist_directory=temp_dir)
retriever = RetrieverFactory.create_retriever(vectorstore, strategy="mmr", top_k=5)

# Build two chains — identical except for the LLM
openai_chain = build_rag_chain(retriever, openai_llm)
anthropic_chain = build_rag_chain(retriever, anthropic_llm)

print("Both chains ready.")

## Side-by-Side Comparison

In [ ]:
question = "What is the difference between similarity search and MMR retrieval?"

# OpenAI
start = time.time()
openai_answer = openai_chain.invoke(question)
openai_time = time.time() - start

# Anthropic
start = time.time()
anthropic_answer = anthropic_chain.invoke(question)
anthropic_time = time.time() - start

print(f"Question: {question}")
print(f"\n{'='*60}")
print(f"OpenAI (gpt-4o-mini) — {openai_time:.2f}s")
print(f"{'='*60}")
print(openai_answer)

print(f"\n{'='*60}")
print(f"Anthropic (Claude Sonnet) — {anthropic_time:.2f}s")
print(f"{'='*60}")
print(anthropic_answer)

## Switching Providers via Environment Variable

In production, you don't change code to switch providers — you change an environment variable:

```bash
# In .env
LLM_PROVIDER=openai      # or "anthropic"
LLM_MODEL=gpt-4o-mini    # or "claude-sonnet-4-20250514"
```

Then your code just calls:
```python
llm = LLMProvider.get_llm()  # reads from config
```

No code changes needed.

## When to Use Which Provider

| Provider | Strengths | Best for |
|----------|----------|----------|
| **OpenAI (GPT-4o-mini)** | Fast, cheap, great function calling | High-volume, cost-sensitive workloads |
| **OpenAI (GPT-4o)** | Strong reasoning, vision | Complex analysis, multimodal |
| **Anthropic (Claude Sonnet)** | Long context (200K tokens), careful responses | Large document Q&A, safety-critical |
| **Anthropic (Claude Haiku)** | Very fast, cheapest | High-throughput, simple tasks |

## Architecture Pattern: The Factory

```python
class LLMProvider:
    @staticmethod
    def get_llm(provider=None, model=None, **kwargs) -> BaseChatModel:
        if provider == "openai":
            return ChatOpenAI(model=model, ...)
        elif provider == "anthropic":
            return ChatAnthropic(model=model, ...)
```

This works because LangChain's `ChatOpenAI` and `ChatAnthropic` both implement `BaseChatModel`. The rest of the pipeline only depends on `BaseChatModel`, not on a specific provider.

**Next:** [08 - Evaluation](./08_evaluation.ipynb) — measuring which provider gives better RAG answers.